## Actividad 3_19: Arroz
<div style="border-style:groove;border-width:thin;padding:10px">
En esta actividad vamos a utilizar las técnicas de redes neuronales y deep learning que hemos visto en clase para solucionar el problema de clasificar tipos de arroz.
    <ol>
        <li>Descarga el archivo "Rice_Image_Dataset.zip" de <a href="https://www.muratkoklu.com/datasets/vtdhnd09.php">https://www.muratkoklu.com/datasets/vtdhnd09.php</a>. Descomprime el archivo y guarda la carpeta en un lugar adecuado.</li>
        <li>Importa los datos usando las misma técnica que en la actividad de los pistachos (En este caso, 250x250 en escala de grises).</li>
        <li>Guarda las etiquetas de los datos. Debes tener un conjunto photos y otro labels que estén ordenados igual. labels debe tener números entre 0.0 y 4.0, ya que hay 5 clases de arroz.</li>
        <li>Utiliza PCA para reducir el dataset.</li>
        <li>Soluciona el ejercicio con una red neuronal. Puedes utilizar todas las técnicas que hemos aprendido.</li>
        <li>Soluciona el ejercicio usando una red convolucional.</li>
    </ol>
</div>

In [1]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
#Deshabilitar la GPU:
#tf.config.set_visible_devices([], 'GPU')
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

folders = listdir('Rice_Image_Dataset')
#Clase Arborio será la clase 0.0
#Clase Basmati será la clase 1.0
#Clase Ipsala será la clase 2.0
#Clase Jasmine será la clase 3.0
#Clase Karacadag será la clase 4.0

photos =  []
labels = []


I0000 00:00:1776179280.166868   12378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776179280.233516   12378 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776179281.705350   12378 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
#2. IMPORTAMOS LOS DATOS:
#Cojo solo 1000 imágenes de cada tipo para ahorrar memoria RAM y tiempo.
for idx,folder in enumerate(folders):
    for file in listdir('Rice_Image_Dataset/'+folder)[:1000]:
        #Cargamos la imagen.
        photo = load_img('Rice_Image_Dataset/'+folder+'/' + file, target_size=(250, 250), color_mode='grayscale') #Cogemos en escala de grises para ahorrar memoria.
        #Convertimos la imagen a un array.
        photo = img_to_array(photo)
        #Los guardamos en las listas.
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print(idx)

0
1
2
3
4


In [3]:
#TRANSFORMAMOS LOS DATOS A NUMPY ARRAY.
#Es necesario para el reshape y, en general, le suele gustar más a los algoritmos.
photos_array = asarray(photos)
labels_array = asarray(labels)

In [4]:
#VEAMOS QUE PINTA TIENEN LOS DATOS
print(photos_array.shape, labels_array.shape)

(5000, 250, 250, 1) (5000,)


Como podéis observar, tenemos 5000 datos que son matrices de 250x250 cada una. Esto tenemos que manejarlo. Para la red neuronal normal necesitamos datos planos (en forma de lista de una sola dimensión) y para la red convolucional, los datos originales, los datos en forma de matriz.

In [5]:

#Vamos a aplanar los datos para que cada entrada tenga una sola dimensión (la X tendrá 2 dimensiones, cada fila 1 dimensión) 
#Esto lo hacemos porque lo necesita la red neuronal normal. También se podría poner lo primero una capa Flatten.
photos_reshape = photos_array.reshape(photos_array.shape[0],-1) 

In [6]:
photos_reshape.shape

(5000, 62500)

In [7]:
#4. UTILIZAMOS PCA PARA SIMPLIFICAR EL CONJUNTO DE DATOS:
#No lo voy a ejecutar porque tarda mucho y porque, para
#este ejerccicio en concreto tiene más sentido usar los datos
#originales.
from sklearn.decomposition import PCA
pca = PCA(n_components=0.95)
X = pca.fit_transform(photos_reshape)
y = labels_array

In [8]:
#Asigno, por tanto, X e y a los datos originales.
X = photos_reshape
y = labels_array

Si habéis hecho pca, veréis que tarda un montón. Te devuelve 438 (si mal no recuerdo) componentes. Pasamos de 62500 a 438. El problema es que para la convolucional, hemos perdido las matrices originales. Una posible solución es forzarle a que tenga forma matricial, pero no tengo muy claro que esto vaya a ser bueno para la convolucional. Imaginad el dibujo que quedaría usando una matriz, digamos de 625 componentes (25x25). La imagen podría no tener nada que ver con la original. En los ejemplos que podréis encontrar por ahí se usa PCA y después se reconstruye la imagen. Eso para nosotros tendría poco sentido porque la imagen reconstruida tiene el tamaño original.
Teniendo en cuenta que la convolucional lo que busca es bordes y demás, esta técnica no nos sirve. Habría que buscar otras.

In [9]:
#Vamos a solucionarlo con el conjunto completo
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [10]:
print(X_train.shape, y_train.shape)

(4500, 62500) (4500,)


In [11]:
#Lo solucionamos con RandomForest para establecer una comparación con los resultados que obtendremos mediante red neuronal.
from sklearn.ensemble import RandomForestClassifier
rnd_clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
rnd_clf.fit(X_train,y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
#Veamos que tal lo ha hecho con random forest
from sklearn.metrics import accuracy_score
y_pred = rnd_clf.predict(X_test)
print(accuracy_score(y_test,y_pred))


0.952


Ya veis, por el resultado, que el randomForest funciona muy muy bien. Será dificil mejorarlo.

In [13]:
#5. SOLUCIÓN CON UNA RED NEURONAL CONVENCIONAL:
#Ahora creamos una red neuronal. Para ello usamos algunas técnicas que hemos visto, como los inicilializadores o la normalización entre capas.
#Este ejercicio es multiclass, no multilabel. No puede haber un arroz de dos tipos a la vez.
from tensorflow import keras
model = keras.models.Sequential()
model.add(keras.layers.InputLayer(shape=X_train.shape[1:]))
model.add(keras.layers.Dense(300,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(200,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(20,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.BatchNormalization())
model.add(keras.layers.Dense(5,activation='softmax',kernel_initializer='glorot_normal'))

E0000 00:00:1776179980.656451   12378 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1776179980.656711   22360 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1776179980.695202   12378 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [ ]:
#Vamos a usar Adam como optimizador y entrenamos.
model.compile(loss='crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100000,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/100000


I0000 00:00:1746833857.349616   25418 service.cc:145] XLA service 0x771fc8012b30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1746833857.349685   25418 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 2070 with Max-Q Design, Compute Capability 7.5
2025-05-10 01:37:37.468830: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-05-10 01:37:37.903464: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907


 5/16 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.4722 - loss: 1.4157

I0000 00:00:1746833860.320963   25418 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


16/16 ━━━━━━━━━━━━━━━━━━━━ 11s 292ms/step - accuracy: 0.6877 - loss: 0.9331 - val_accuracy: 0.5178 - val_loss: 2.5763
Epoch 2/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9476 - loss: 0.2962 - val_accuracy: 0.4667 - val_loss: 1.9451
Epoch 3/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - accuracy: 0.9697 - loss: 0.2215 - val_accuracy: 0.4956 - val_loss: 1.6321
Epoch 4/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9697 - loss: 0.1928 - val_accuracy: 0.4978 - val_loss: 1.4026
Epoch 5/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9765 - loss: 0.1717 - val_accuracy: 0.5089 - val_loss: 1.3331
Epoch 6/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.9871 - loss: 0.1468 - val_accuracy: 0.5511 - val_loss: 1.1572
Epoch 7/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - accuracy: 0.9875 - loss: 0.1317 - val_accuracy: 0.6822 - val_loss: 0.8394
Epoch 8/100000
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.9896 - loss: 0.1207 - val_a

In [18]:
#Finalmente evaluamos el resultado con el conjunto de test.
model.evaluate(X_test,y_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.9439 - loss: 0.1910


[0.1626472771167755, 0.9559999704360962]

Vemos que anda en una línea muy parecida al RandomForest. Vamos a probar con una convolucional.

In [19]:
#6. SOLUCIONAMOS EL PROBLEMA CON UNA CONVOLUCIONAL.
#Para la covolucional nos quedamos con los datos originales. Tienen que estar en forma de matriz.
X = photos_array/255.0
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = 0)

In [18]:
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPool2D
from tensorflow.keras.layers import Flatten
model = keras.models.Sequential()
model.add(Conv2D(16,(3,3), activation='relu', input_shape = (250,250,1)))
model.add(MaxPool2D(2,2))
model.add(Conv2D(32,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Conv2D(64,(3,3), activation='relu'))
model.add(MaxPool2D(2,2))
model.add(Flatten())
model.add(keras.layers.Dense(100,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(30,activation='relu',kernel_initializer='he_normal'))
model.add(keras.layers.Dense(5,activation='softmax',kernel_initializer='glorot_normal'))

In [21]:
#Vamos a usar Adam como optimizador y entrenamos.
model.compile(loss='crossentropy', optimizer = keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.9, beta_2=0.999), metrics=['accuracy'])
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,
restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=34,validation_split = 0.1,callbacks=[early_stopping_cb],batch_size=256)

Epoch 1/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 31s 1s/step - accuracy: 0.3037 - loss: 1.4390 - val_accuracy: 0.4556 - val_loss: 1.0365
Epoch 2/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 154ms/step - accuracy: 0.6065 - loss: 0.8990 - val_accuracy: 0.9178 - val_loss: 0.5506
Epoch 3/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.9187 - loss: 0.4445 - val_accuracy: 0.9200 - val_loss: 0.2788
Epoch 4/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.9222 - loss: 0.2440 - val_accuracy: 0.9156 - val_loss: 0.2263
Epoch 5/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.9385 - loss: 0.1868 - val_accuracy: 0.9422 - val_loss: 0.1673
Epoch 6/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.9608 - loss: 0.1342 - val_accuracy: 0.9511 - val_loss: 0.1469
Epoch 7/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 155ms/step - accuracy: 0.9611 - loss: 0.1214 - val_accuracy: 0.9467 - val_loss: 0.1368
Epoch 8/34
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 156ms/step - accuracy: 0.9661 - loss: 0.1077 - val_accuracy: 0.96

In [22]:
#Finalmente evaluamos el resultado con el conjunto de test.
model.evaluate(X_test,y_test)

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 56ms/step - accuracy: 0.9538 - loss: 0.1395


[0.10506893694400787, 0.9660000205039978]

Vemos que el resultado mejora un poco pero no mucho. Esto se debe a que los arroces están en el centro de las imágenes. La verdadera potencia de las convolucionales es cuando tenemos los objetos distribuidos por las imágenes.